# One pass per question: all options in one input (listwise Signpost)

**Experiment** (branch `exp/listwise-index`, run 26), cut from `signpost`. Signpost (`03_lora_signpost.ipynb`) reads the conversation once **per option**: 15 agents means 15 encoder passes. Here every option goes into **one** input, the encoder runs **once**, and one shared head scores each option at its own tokens. The answer is the option's number, 1…N:

```text
<history> [SEP] current agent: X [SEP] question: Q [SEP] option: Billing Agent. Handles… [SEP] option: Returns Agent. … [SEP] …
   → ModernBERT-large + LoRA (one pass) → the mean of each option's tokens → the same head on each → softmax over the N options → 1…N
```

The options also see each other inside the encoder, so the model can compare them.

**What changes from Signpost**
- `MAX_LEN` 1,536 tokens, enough for a question with 50 options or a chat with 15 agents. If anything is too long, the oldest history is dropped; options never are.
- Routing training adds **distractor agents** from other businesses, up to 15 per question. Agents with generic names, or names sharing a word with the business's own agents, are never added, so there are no near-duplicates.
- **Training data, about 10.5K questions:**
  - all of Signpost's yes/no questions;
  - 30% of its routing conversations and 30% of its choice and score questions, sampled evenly across businesses and families;
  - **2,500 hand-written many-option questions:** 957 individually written texts, with 2–50 options.
- **Training batches by token budget** (similar lengths together).

**Tests**
- **The old suites** (E10, E11, banking, E12, independent), for a direct comparison with Signpost run 25.
- **Two fresh tests:**
  - **many-option questions:** 162 hand-written questions with 10–50 options, using words and names that never appear in training;
  - **many-agent routing:** 12 new businesses with 9–15 agents, 24 conversations (120 turns, routed live) and 72 first messages.
- An **option-order test:** the same first messages with the options reversed; the answer shouldn't change.

On the fresh tests, the published **Signpost** (downloaded from the Hub) and the **Strands decider** run in this notebook too, so all three models are compared on the same items, with latency.

## 1 — Setup

| Setting | Default | Meaning |
|---|---|---|
| `MAX_LEN` | 1536 | tokens per input (history + question + all options) |
| `MAX_OPTIONS` | 50 | more options than this raises an error |
| `AUG_MAX_AGENTS` | 15 | routing questions are trained with between their own 5 agents and this many |
| `ROUTE_FRACTION`, `QUESTION_FRACTION` | 0.3, 0.3 | share of Signpost's routing and choice/score training data used (all yes/no and all many-option questions are used) |
| `TOKEN_BUDGET`, `MAX_BATCH` | 8000, 16 | a training batch holds about this many tokens (padding included), at most this many questions |
| `EPOCHS` | 2 | passes over all training questions |
| `LORA_R`, `LORA_ALPHA`, `LR`, `HEAD_LR` | 16, 32, 2e-4, 1e-3 | as in Signpost |
| `SCORE_LOSSES` | `ordinal` | the score loss adopted in run 21 |
| `RUN_SIGNPOST_HUB`, `RUN_DECIDER` | 1, 1 | also run the published Signpost and the decider on the new tests |
| `LORA_SMOKE` | 0 | 1 = a tiny code check on a CPU |

Runs on Kaggle with **2× T4**: our model on `cuda:0`, the decider on `cuda:1`. With one GPU everything runs on `cuda:0`, one model after another.

In [ ]:
!pip uninstall -y -q torchao
!pip install -q "transformers==5.17.0" "strands-decider"
import copy, gc, json, math, os, random, statistics, subprocess, time
from collections import Counter
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F

SMOKE = os.environ.get("LORA_SMOKE") == "1"
RUN_ENCODER = os.environ.get("RUN_ENCODER", "modernbert-base" if SMOKE else "modernbert-large")
# "combined" trains one model on routing and questions together (Signpost); "separate" trains two independent models from
# scratch, one on routing only and one on the questions only, each scored on its own tests (run 24).
TRAIN_MODE = os.environ.get("TRAIN_MODE", "combined")
LORA_R, LORA_ALPHA, LORA_DROPOUT = int(os.environ.get("LORA_R", "16")), float(os.environ.get("LORA_ALPHA", "32")), 0.1
EPOCHS = int(os.environ.get("EPOCHS", "1" if SMOKE else "2"))
LR, HEAD_LR = float(os.environ.get("LR", "2e-4")), float(os.environ.get("HEAD_LR", "1e-3"))
MAX_LEN = int(os.environ.get("MAX_LEN", "1536"))                  # history + question + every option, in one input
MAX_OPTIONS = int(os.environ.get("MAX_OPTIONS", "50"))
AUG_MAX_AGENTS = int(os.environ.get("AUG_MAX_AGENTS", "15"))      # routing questions trained with 5..this many agents
TOKEN_BUDGET, MAX_BATCH = int(os.environ.get("TOKEN_BUDGET", "8000")), int(os.environ.get("MAX_BATCH", "16"))
ROUTE_FRACTION = float(os.environ.get("ROUTE_FRACTION", "0.3"))          # share of Signpost's routing training data
QUESTION_FRACTION = float(os.environ.get("QUESTION_FRACTION", "0.3"))    # share of its choice and score questions
RUN_SIGNPOST_HUB = os.environ.get("RUN_SIGNPOST_HUB", "1") == "1"
RUN_DECIDER = os.environ.get("RUN_DECIDER", "1") == "1"
# The score-question loss. "plain" = cross-entropy on the true level; "ordinal" = SCORE_EPS of the target spread onto the
# neighbouring levels + SCORE_DIST_W x |expected level - true level| (adopted in run 21). Each listed variant is trained in turn.
SCORE_LOSSES = os.environ.get("SCORE_LOSSES", "ordinal").split(",")
SCORE_EPS, SCORE_DIST_W = float(os.environ.get("SCORE_EPS", "0.1")), float(os.environ.get("SCORE_DIST_W", "0.5"))
SEED = int(os.environ.get("SEED", "42"))
K_MAX = MAX_OPTIONS                                      # (used by the base class's per-option scorer)
ROUTE_Q = "Which agent should handle the user's latest message?"

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
DECIDER_DEVICE = os.environ.get("DECIDER_DEVICE", "cuda:1" if torch.cuda.device_count() > 1 else "cuda:0")
AMP = DEVICE.type == "cuda"
RUN_LOG = Path("listwise_run.log")


def log(message):
    line = f"[{time.strftime('%H:%M:%S')}] {message}"
    print(line, flush=True)
    with open(RUN_LOG, "a") as fh:
        fh.write(line + "\n")


def gpu():
    return f"GPU {torch.cuda.memory_allocated(DEVICE) / 1e9:.1f} GB (peak {torch.cuda.max_memory_allocated(DEVICE) / 1e9:.1f})" if AMP else "CPU"


import transformers
log(f"torch {torch.__version__} | transformers {transformers.__version__} | device {DEVICE}"
    + (f" ({torch.cuda.get_device_name(DEVICE)})" if AMP else "") + f" | encoder {RUN_ENCODER} | decider on {DECIDER_DEVICE}" + (" | SMOKE CHECK" if SMOKE else ""))

## 2 — Data

**Training mix:**
- all yes/no questions;
- `ROUTE_FRACTION` of the routing conversations and `QUESTION_FRACTION` of the choice and score questions, sampled evenly per business or question family;
- the 2,500 hand-written many-option questions.

The dev sets are Signpost's, so dev numbers stay comparable. Routing questions keep their own 5 agents here; distractor agents are added during training, a different random set each time. The fresh tests are loaded at the end.

In [ ]:
DATA_BRANCH = "exp/listwise-index"                          # this notebook's branch (cloned on Kaggle / Colab)
DATA_DIR = Path("data/modernbert_router")
if not DATA_DIR.exists():
    repo = Path("llms-the-hard-way")
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", DATA_BRANCH,
                        "https://github.com/skamalj/llms-the-hard-way.git", str(repo)], check=True)
    else:
        subprocess.run(["git", "-C", str(repo), "fetch", "--depth", "1", "origin", DATA_BRANCH], check=True)
        subprocess.run(["git", "-C", str(repo), "checkout", "-B", DATA_BRANCH, "FETCH_HEAD"], check=True)
    DATA_DIR = repo / "data" / "modernbert_router"
load = lambda p: json.load(open(DATA_DIR / p))

# ---- routing training data
domains = load("generic/domains.json"); domains.update(load("realistic_train/domains.json"))
route_convs = load("generic/conversations.json") + load("realistic_train/conversations.json")
route_items = [{"task": "routing", "domain": ex["domain"], "messages": ex["messages"], "current": ex["current_agent"],
                "question": ROUTE_Q, "options": domains[ex["domain"]]["agents"], "answer": ex["target_agent"]} for ex in route_convs]
# ---- choice training data
choice_items = [{"task": it["type"], "domain": it["domain"], "messages": [{"role": "user", "content": it["state"]}], "current": None,
                 "question": it["question"], "options": {o: "" for o in it["options"]}, "answer": it["answer"], "state": it["state"],
                 "levels": list(it["options"])}            # for score questions: the levels from low to high
                for it in load("choice_train/items.json")["items"] + load("many_train/items.json")["items"]]   # + many-option family

rng = random.Random(SEED)
route_domains = sorted({x["domain"] for x in route_items}); rng.shuffle(route_domains)
dev_domains = set(route_domains[: max(1, len(route_domains) // 10)])
texts = sorted({x["state"] for x in choice_items}); rng.shuffle(texts)
dev_texts = set(texts[: len(texts) // 10])
train_route = [x for x in route_items if x["domain"] not in dev_domains]
dev_route = [x for x in route_items if x["domain"] in dev_domains]
train_choice = [x for x in choice_items if x["state"] not in dev_texts]
dev_choice = [x for x in choice_items if x["state"] in dev_texts]

# ---- tests
real_convs = load("realistic/conversations.json")
ft = load("first_turn/messages.json")
banking = load("banking/benchmark.json") + load("banking/extended.json")
e12 = load("choice_test/items.json")["items"]
BANKING_AGENTS = {                                          # exactly as in 02_frozen_router.ipynb, Cell 37
    "Payments Agent": "Handles bank transfers, failed or pending payments, transaction status, payment charges and "
                      "duplicate charges, refunds for transactions, and settlement problems where money was sent but "
                      "not received. Does not handle card servicing such as blocking, replacing or activating cards.",
    "Card Support Agent": "Handles debit and credit card servicing: card activation, blocking and unblocking, lost or "
                          "stolen cards, card replacement and delivery, card limits, card PIN, card security and "
                          "compromised card details, and card-specific transactions.",
    "Investment Agent": "Handles mutual funds, SIPs, investments, portfolio allocation, investment transactions, "
                        "redemptions and investment planning.",
    "Insurance Agent": "Handles insurance policies, premiums, renewals, claims, policy expiry and policy servicing.",
    "General Customer Support Agent": "Handles general account questions, profile and contact details, issues that do "
                                      "not clearly belong to a specialist, and routing customers to the right agent.",
}
# ---- independent test set (written outside the project; see experiments/independent_test_brief.md)
ind_q = load("independent/questions.json")["items"]
ind_r = load("independent/routing.json")
ind_f = load("independent/first_turn.json")["messages"]
companies = sorted({c["company"] for c in real_convs})
TEST_COMPANIES = set(companies) - set(companies[::2])           # the same 6 test companies as 02_frozen_router.ipynb, Cell 70

# ---- fresh tests: many options, many agents
many_q = load("many_test/questions.json")["items"]
many_r = load("many_test/routing.json")
many_f = load("many_test/first_turn.json")["messages"]

# ---- distractor agents for routing training: other businesses' agents, never generic ones or ones sharing a name word
GENERIC = {"general", "support", "help", "customer", "customers", "billing", "payment", "payments", "account", "accounts",
           "booking", "bookings", "enquiries", "service", "services", "care", "desk", "team"}


def name_words(name):
    return {w.strip(",.&'").lower() for w in name.split()} - {"agent", "and", "&", ""}


DISTRACTORS = [(n, d, dom) for dom, info in domains.items() for n, d in info["agents"].items() if not name_words(n) & GENERIC]


def add_distractors(item, rng, n_total):
    """Item with its own agents plus (n_total - own) agents from other businesses; no shared name words with its own."""
    own = set().union(*(name_words(n) for n in item["options"]))
    pool = [(n, d) for n, d, dom in DISTRACTORS if dom != item["domain"] and not name_words(n) & own]
    extra = rng.sample(pool, max(0, min(len(pool), n_total - len(item["options"]))))
    return {**item, "options": {**item["options"], **dict(extra)}}


def stratified(items, key, frac, rng):
    """`frac` of the items from every group (at least one per group), so coverage stays as wide as the full set."""
    groups = {}
    for x in items:
        groups.setdefault(x[key], []).append(x)
    return [x for g in groups.values() for x in rng.sample(g, max(1, round(frac * len(g))))]


_mix_rng = random.Random(SEED)
_full = {"routing": len(train_route), "questions": len(train_choice)}
train_route = stratified(train_route, "domain", ROUTE_FRACTION, _mix_rng)
_many = [x for x in train_choice if x["domain"].startswith("many/")]
_yes_no = [x for x in train_choice if x["task"] == "noul"]
_other = [x for x in train_choice if x["task"] in ("choice", "score") and not x["domain"].startswith("many/")]
train_choice = _yes_no + stratified(_other, "domain", QUESTION_FRACTION, _mix_rng) + _many
print(f"training mix: routing {len(train_route)} of {_full['routing']} | yes/no {len(_yes_no)} (all) | choice + score "
      f"{len(train_choice) - len(_yes_no) - len(_many)} of {len(_other)} | many-option {len(_many)} | total {len(train_route) + len(train_choice)}")

_dev_rng = random.Random(SEED)
dev_route_many = [add_distractors(x, _dev_rng, AUG_MAX_AGENTS) for x in dev_route]     # fixed, for the training log

if SMOKE:
    train_route, dev_route, train_choice, dev_choice = train_route[::400], dev_route[::100], train_choice[::400], dev_choice[::100]
    real_convs, banking, e12 = real_convs[:2], banking[:6], e12[::20]
    many_q, many_r, many_f, dev_route_many = many_q[::40], many_r[:2], many_f[::12], dev_route_many[::100]
    ft = {"companies": ft["companies"], "messages": ft["messages"][::20]}

for name, xs in [("train routing", train_route), ("train choice", train_choice), ("dev routing", dev_route), ("dev choice", dev_choice)]:
    print(f"{name:14s}: {len(xs):5d} questions | " + ", ".join(f"{t} {n}" for t, n in Counter(x["task"] for x in xs).items()))
print(f"dev routing domains (unseen in training): {sorted(dev_domains)}")
print(f"independent: questions {len(ind_q)} | routing {len(ind_r)} conversations / {sum(len(c['user_turns']) for c in ind_r)} turns "
      f"(flagged: {dict(Counter(u['flag'] for c in ind_r for u in c['user_turns'] if u['flag']))}) | first turn {len(ind_f)}")
print(f"distractor agents available: {len(DISTRACTORS)} | many-option questions in training: "
      f"{sum(x['domain'].startswith('many/') for x in choice_items)}")
print(f"fresh tests: many-option questions {len(many_q)} ({min(i['n_options'] for i in many_q)}-{max(i['n_options'] for i in many_q)} options) | "
      f"many-agent routing {len(many_r)} conversations / {sum(len(c['user_turns']) for c in many_r)} turns | first messages {len(many_f)}")
print(f"tests: E10 {len(real_convs)} conversations / {sum(len(c['user_turns']) for c in real_convs)} turns | E11 {len(ft['messages'])} | "
      f"banking {len(banking)} | E12 {len(e12)}")

## 3 — The model: one input, one pass, one head per option

The encoder, LoRA and head are Signpost's (`LoRALinear` written by hand, r 16 on every attention and MLP layer; head `Linear(H, 256) → GELU → Linear(256, 1)`). What changes is `ListwiseModel.scores`:
1. `fit` builds one text per question, with the options last, each marked `option:`. It records each option's **character span**, and drops the oldest messages until the text fits `MAX_LEN`.
2. One encoder pass over the batch. The tokenizer's offset mapping tells which tokens belong to which option.
3. Each option's vector is the **mean of its tokens**; the same head scores every option.
4. Missing options (questions with fewer options than others in the batch) get −1e4, so the softmax ignores them.

In [ ]:
from transformers import AutoModel, AutoTokenizer

ENCODERS = {
    "modernbert-large": {"name": "answerdotai/ModernBERT-large", "sep": " [SEP] ", "prefix": "", "dtype": torch.float32,
                         "targets": {"Wqkv", "Wo", "Wi"}, "remote": False, "ckpt": False},
    "modernbert-base":  {"name": "answerdotai/ModernBERT-base", "sep": " [SEP] ", "prefix": "", "dtype": torch.float32,
                         "targets": {"Wqkv", "Wo", "Wi"}, "remote": False, "ckpt": False},
    "nemotron-1b":      {"name": "nvidia/llama-nemotron-embed-1b-v2", "sep": "\n", "prefix": "query: ", "dtype": torch.float16,
                         "targets": {"q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"},
                         "remote": True, "ckpt": True},
}


class LoRALinear(nn.Module):
    """A frozen linear layer plus a trainable low-rank update: y = W x + (alpha / r) * B (A x)."""

    def __init__(self, base, r, alpha, dropout):
        super().__init__()
        self.base = base
        self.A = nn.Parameter(torch.empty(r, base.in_features, device=base.weight.device))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r, device=base.weight.device))
        self.scale, self.drop = alpha / r, nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + (self.drop(x) @ self.A.t().to(x.dtype) @ self.B.t().to(x.dtype)) * self.scale


def add_lora(model, targets):
    found = [(m, n, c) for m in list(model.modules()) for n, c in list(m.named_children()) if n in targets and isinstance(c, nn.Linear)]
    for m, n, c in found:
        setattr(m, n, LoRALinear(c, LORA_R, LORA_ALPHA, LORA_DROPOUT))
    return len(found)


class Model:
    """One encoder with LoRA + one scoring head, and everything needed to score a question."""

    def __init__(self, key):
        self.key, self.spec = key, ENCODERS[key]
        spec = self.spec
        self.tok = AutoTokenizer.from_pretrained(spec["name"], trust_remote_code=spec["remote"])
        self.tok.padding_side, self.tok.truncation_side = "right", "left"
        if self.tok.pad_token is None:
            self.tok.pad_token = self.tok.eos_token
        self.enc = AutoModel.from_pretrained(spec["name"], trust_remote_code=spec["remote"], dtype=spec["dtype"]).to(DEVICE)
        self.enc.requires_grad_(False)
        self.n_lora = add_lora(self.enc, spec["targets"])
        assert self.n_lora > 0, f"no LoRA target layers found in {spec['name']}"
        if spec["ckpt"]:
            try:
                self.enc.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
            except Exception as e:                           # some remote-code models do not support it
                log(f"gradient checkpointing not available: {e}")
        self.H = self.enc.config.hidden_size
        self.head = nn.Sequential(nn.Dropout(0.1), nn.Linear(self.H, 256), nn.GELU(), nn.Dropout(0.1), nn.Linear(256, 1)).to(DEVICE)

    def lora_params(self):
        return [p for p in self.enc.parameters() if p.requires_grad]

    def train(self, mode=True):
        self.enc.train(mode); self.head.train(mode)

    def text(self, item, name):
        sep, desc = self.spec["sep"], item["options"][name]
        parts = [f"{m['role']}: {m['content']}" for m in item["messages"]]
        parts += [f"current agent: {item['current'] or 'none'}", f"question: {item['question']}",
                  f"candidate: {name}. {desc}" if desc else f"candidate: {name}"]
        return self.spec["prefix"] + sep.join(parts)

    def scores(self, items):
        """-> [len(items), K_MAX] scores (padded slots -1e4), one batched encoder pass over every option of every item."""
        inputs, owner, slot = [], [], []
        for b, it in enumerate(items):
            for k, name in enumerate(it["options"]):
                inputs.append(self.text(it, name)); owner.append(b); slot.append(k)
        enc = self.tok(inputs, padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt").to(DEVICE)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP):
            hidden = self.enc(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"]).last_hidden_state
        m = enc["attention_mask"].unsqueeze(-1).float()
        s = self.head((hidden.float() * m).sum(1) / m.sum(1)).squeeze(-1)
        out = torch.full((len(items), K_MAX), -1e4, device=DEVICE)
        out[torch.tensor(owner, device=DEVICE), torch.tensor(slot, device=DEVICE)] = s
        return out

    def probs(self, item):
        """Probabilities over one item's options (in order), no gradient."""
        with torch.no_grad():
            return torch.softmax(self.scores([item]), dim=1)[0, :len(item["options"])].float().cpu()


class ListwiseModel(Model):
    """All of a question's options in one input; each option scored from the mean of its own tokens by the shared head."""

    def build(self, item, drop=0):
        """-> (text, [(start, end) character span of each option]), without the `drop` oldest messages."""
        sep = self.spec["sep"]
        parts = [f"{m['role']}: {m['content']}" for m in item["messages"][drop:]]
        parts += [f"current agent: {item['current'] or 'none'}", f"question: {item['question']}"]
        text, spans = self.spec["prefix"] + sep.join(parts), []
        for name, desc in item["options"].items():
            seg = f"{name}. {desc}" if desc else name
            text += sep + "option: "
            spans.append((len(text), len(text) + len(seg)))
            text += seg
        return text, spans

    def fit(self, item):
        """Drop the oldest messages until the input fits MAX_LEN; the latest message and every option always stay."""
        assert len(item["options"]) <= MAX_OPTIONS, f"{len(item['options'])} options > MAX_OPTIONS={MAX_OPTIONS}"
        for drop in range(len(item["messages"])):
            text, spans = self.build(item, drop)
            if len(self.tok(text)["input_ids"]) <= MAX_LEN:
                return text, spans
        raise ValueError(f"{len(item['options'])} options don't fit in MAX_LEN={MAX_LEN}, even with only the latest message")

    def scores(self, items):
        """-> [len(items), max options] scores (missing options -1e4), one encoder pass for the whole batch."""
        built = [self.fit(it) for it in items]
        enc = self.tok([t for t, _ in built], padding=True, return_offsets_mapping=True, return_tensors="pt")
        offsets = enc.pop("offset_mapping")                                   # [B, T, 2] characters of each token
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP):
            hidden = self.enc(input_ids=enc["input_ids"].to(DEVICE), attention_mask=enc["attention_mask"].to(DEVICE)).last_hidden_state
        K, T = max(len(sp) for _, sp in built), hidden.shape[1]
        cs, ce = offsets[..., 0], offsets[..., 1]
        real = enc["attention_mask"].bool() & (ce > cs)                       # real, non-special tokens
        member = torch.zeros(len(items), K, T)
        for b, (_, spans) in enumerate(built):
            for k, (s0, e0) in enumerate(spans):
                member[b, k] = (real[b] & (cs[b] < e0) & (ce[b] > s0)).float()    # tokens overlapping option k
        member = member.to(DEVICE)
        counts = member.sum(-1)                                               # [B, K] tokens per option
        assert all((counts[b, :len(sp)] > 0).all() for b, (_, sp) in enumerate(built)), "an option got no tokens"
        pooled = (member @ hidden.float()) / counts.clamp(min=1).unsqueeze(-1)    # [B, K, H] mean of each option's tokens
        s = self.head(pooled).squeeze(-1)                                     # [B, K]: one shared head for every option
        return s.masked_fill(counts == 0, -1e4)

## 4 — Training and evaluation

As in Signpost (ordinal score loss, collapse guard, best epoch on dev), with two changes in training:
- **Batches by token budget:** questions of similar length go together, up to `TOKEN_BUDGET` tokens.
- **Each time a routing question is used,** it gets a new random number of distractor agents (up to `AUG_MAX_AGENTS`), then all options are shuffled, so position can't be learned.

**New evaluations, on the fresh tests:**
- many-option questions (by family and by option count);
- many-agent routing, live and with the correct previous agent given;
- many-agent first messages;
- **option order:** the same first messages with the options reversed.

In [ ]:
def lat_summary(ms):
    ms = sorted(ms)
    return {"p50": ms[len(ms) // 2], "p95": ms[min(len(ms) - 1, int(0.95 * len(ms)))], "mean": sum(ms) / len(ms), "n": len(ms)}


def timed_probs(model, item):
    if AMP:
        torch.cuda.synchronize(DEVICE)
    t0 = time.perf_counter()
    p = model.probs(item)
    if AMP:
        torch.cuda.synchronize(DEVICE)
    return p, (time.perf_counter() - t0) * 1000


def dev_accuracy(model, items, batch=8):
    model.train(False)
    ok = 0
    with torch.no_grad():
        for s0 in range(0, len(items), batch):
            chunk = items[s0:s0 + batch]
            pick = model.scores(chunk).argmax(1).tolist()
            ok += sum(list(it["options"])[k] == it["answer"] for it, k in zip(chunk, pick))
    return ok / max(1, len(items)), ok


def ordinal_loss(scores, items, target):
    """Listwise loss where score questions get an order-aware target: SCORE_EPS of the probability goes to the neighbouring
    levels (near-misses cost less), plus SCORE_DIST_W x |expected level - true level| / (levels - 1) (far misses cost more).
    Yes / no, choice and routing questions keep the plain one-hot target."""
    logp = F.log_softmax(scores, dim=1)
    p = logp.exp()
    soft = torch.zeros_like(scores)
    dist = []
    for b, it in enumerate(items):
        if it["task"] != "score":
            soft[b, target[b]] = 1.0
            continue
        names = list(it["options"])                              # the (shuffled) order of the slots
        level = [it["levels"].index(n) for n in names]           # the scale position of each slot
        true = it["levels"].index(it["answer"])
        nbrs = [k for k, l in enumerate(level) if abs(l - true) == 1]
        soft[b, target[b]] = 1.0 - (SCORE_EPS if nbrs else 0.0)
        for k in nbrs:
            soft[b, k] = SCORE_EPS / len(nbrs)
        expected = (p[b, :len(level)] * torch.tensor(level, dtype=p.dtype, device=p.device)).sum()
        dist.append((expected - true).abs() / (len(level) - 1))
    loss = -(soft * logp).sum(1).mean()
    return loss + SCORE_DIST_W * torch.stack(dist).mean() if dist else loss


def est_tokens(it):
    """Rough token count before tokenizing (4 characters a token), routing questions at their largest (AUG_MAX_AGENTS agents)."""
    chars = sum(len(m["content"]) + 12 for m in it["messages"]) + len(it["question"]) + 40
    chars += sum(len(n) + len(d) + 12 for n, d in it["options"].items())
    if it["task"] == "routing":
        chars += max(0, AUG_MAX_AGENTS - len(it["options"])) * 130
    return min(MAX_LEN, chars // 4 + 8)


def make_batches(items):
    """Questions of similar length together; a batch holds at most TOKEN_BUDGET tokens (padding included), MAX_BATCH questions."""
    batches, cur, longest = [], [], 0
    for it in sorted(items, key=est_tokens):
        n = est_tokens(it)
        if cur and (max(longest, n) * (len(cur) + 1) > TOKEN_BUDGET or len(cur) >= MAX_BATCH):
            batches.append(cur)
            cur, longest = [], 0
        cur.append(it)
        longest = max(longest, n)
    return batches + ([cur] if cur else [])


class Collapsed(Exception):
    """Training collapsed (run 22: the choice loss stayed at chance after the warm-up peak); retry with another seed."""


COLLAPSE_DEV = {"routing": 0.95, "choice": 0.85}       # healthy runs 19-21 had >= 0.977 / 0.949 after epoch 1


def train_model(model, score_loss="plain", seed=SEED, tasks="both"):
    """tasks: "both" (one model for routing + questions), "routing" or "questions" (a specialist; the other data is not used at all)."""
    use_r, use_c = tasks in ("both", "routing"), tasks in ("both", "questions")
    batches = make_batches(train_route) if use_r else []
    batches += make_batches(train_choice) if use_c else []
    params = model.lora_params() + list(model.head.parameters())
    opt = torch.optim.AdamW([{"params": model.lora_params(), "lr": LR}, {"params": model.head.parameters(), "lr": HEAD_LR}], weight_decay=0.01)
    total = EPOCHS * len(batches)
    warm = max(1, int(0.06 * total))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min((s + 1) / warm, max(0.0, (total - s) / max(1, total - warm))))
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    best, step, t0 = (-1.0, None, 0), 0, time.perf_counter()
    log(f"[{model.key}] score loss: {score_loss}" + (f" (eps {SCORE_EPS}, distance weight {SCORE_DIST_W})" if score_loss == "ordinal" else ""))
    log(f"[{model.key}] training ({tasks}): {len(batches)} batches per epoch ({len(train_route) if use_r else 0} routing + "
        f"{len(train_choice) if use_c else 0} choice questions), {total} steps")
    for epoch in range(1, EPOCHS + 1):
        model.train(True)
        order = list(range(len(batches)))
        random.Random(seed + epoch).shuffle(order)
        aug_rng = random.Random(seed * 7 + epoch)
        run = {"routing": [], "choice": []}
        for bi in order:
            batch = batches[bi]
            items = [add_distractors(it, aug_rng, aug_rng.randint(len(it["options"]), AUG_MAX_AGENTS))
                     if it["task"] == "routing" else dict(it) for it in batch]          # a new set of extra agents every time
            for it in items:                                    # option order shuffled, so position can't be learned
                names = list(it["options"]); random.shuffle(names); it["options"] = {n: it["options"][n] for n in names}
            target = torch.tensor([list(it["options"]).index(it["answer"]) for it in items], device=DEVICE)
            out = model.scores(items)
            loss = ordinal_loss(out, items, target) if score_loss == "ordinal" else F.cross_entropy(out, target)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            run["routing" if batch[0]["task"] == "routing" else "choice"].append(loss.item())
            step += 1
            if step % 100 == 0 or step == total:
                el = time.perf_counter() - t0
                log(f"  [{model.key}] epoch {epoch} step {step}/{total} | loss routing {statistics.mean(run['routing'][-50:] or [0]):.4f} "
                    f"choice {statistics.mean(run['choice'][-50:] or [0]):.4f} | {step / el:.2f} steps/s | "
                    f"~{(total - step) / (step / el) / 60:.1f} min left | {gpu()}")
        accs, msg = [], []
        if use_r:
            acc_r, ok_r = dev_accuracy(model, dev_route); accs.append(acc_r); msg.append(f"dev routing {ok_r}/{len(dev_route)} ({acc_r:.1%})")
            acc_m, ok_m = dev_accuracy(model, dev_route_many)
            msg.append(f"dev routing with {AUG_MAX_AGENTS} agents {ok_m}/{len(dev_route_many)} ({acc_m:.1%})")
        if use_c:
            acc_c, ok_c = dev_accuracy(model, dev_choice); accs.append(acc_c); msg.append(f"dev choice {ok_c}/{len(dev_choice)} ({acc_c:.1%})")
        log(f"[{model.key}] epoch {epoch} done | " + " | ".join(msg))
        if epoch == 1 and ((use_r and acc_r < COLLAPSE_DEV["routing"]) or (use_c and acc_c < COLLAPSE_DEV["choice"])):
            raise Collapsed(" | ".join(msg) + f" after epoch 1 (thresholds {COLLAPSE_DEV})")
        if sum(accs) / len(accs) > best[0]:
            best = (sum(accs) / len(accs), {"lora": {n: p.detach().clone() for n, p in model.enc.named_parameters() if p.requires_grad},
                                          "head": copy.deepcopy(model.head.state_dict())}, epoch)
    params_by_name = dict(model.enc.named_parameters())
    for n, v in best[1]["lora"].items():
        params_by_name[n].data.copy_(v)
    model.head.load_state_dict(best[1]["head"])
    minutes = (time.perf_counter() - t0) / 60
    log(f"[{model.key}] training done in {minutes:.1f} min | best epoch {best[2]} (mean dev accuracy {best[0]:.1%})")
    return {"train_minutes": minutes, "best_epoch": best[2]}


TURN_TYPES = ["start", "answer", "follow", "aspect", "closing", "also_stay", "switch", "short_switch", "return", "handoff_ok", "branch"]


def eval_e10(model):
    model.train(False)
    rows, lat = [], []
    for conv in real_convs:
        cur = None
        for u in conv["user_turns"]:
            prefix = conv["messages"][:u["message_index"] + 1]
            item = {"messages": prefix, "current": cur, "question": ROUTE_Q, "options": conv["agents"]}
            p, ms = timed_probs(model, item)
            lat.append(ms)
            cur = list(conv["agents"])[int(p.argmax())]
            rows.append({"company": conv["company"], "type": u["turn_type"], "ok": cur == u["expected_agent"]})
    test = [r for r in rows if r["company"] in TEST_COMPANIES]
    print(f"E10 live routing | all {sum(r['ok'] for r in rows)}/{len(rows)} | test companies {sum(r['ok'] for r in test)}/{len(test)}")
    print(f"  {'turn type':14s} | {'all':>9} | {'test':>9}")
    for tt in TURN_TYPES:
        a, t_ = [r for r in rows if r["type"] == tt], [r for r in test if r["type"] == tt]
        if a:
            print(f"  {tt:14s} | {sum(r['ok'] for r in a):>3} / {len(a):<4} | {sum(r['ok'] for r in t_):>3} / {len(t_):<4}")
    return {"all": sum(r["ok"] for r in rows), "all_n": len(rows), "test": sum(r["ok"] for r in test), "test_n": len(test),
            "latency": lat_summary(lat)}


def eval_e11(model):
    model.train(False)
    top1 = top2 = 0
    lat = []
    for m in ft["messages"]:
        agents = ft["companies"][m["company"]]
        item = {"messages": [{"role": "user", "content": m["message"]}], "current": None, "question": ROUTE_Q, "options": agents}
        p, ms = timed_probs(model, item)
        lat.append(ms)
        names = list(agents)
        top1 += names[int(p.argmax())] == m["target_agent"]
        top2 += m["target_agent"] in [names[i] for i in p.topk(2).indices.tolist()]
    print(f"E11 first turn | top-1 {top1}/{len(ft['messages'])} | top-2 {top2}/{len(ft['messages'])}")
    return {"top1": top1, "top2": top2, "n": len(ft["messages"]), "latency": lat_summary(lat)}


def eval_banking(model):
    model.train(False)
    ok, lat = [], []
    for ex in banking:
        item = {"messages": ex["messages"], "current": ex["current_agent"], "question": ROUTE_Q, "options": BANKING_AGENTS}
        p, ms = timed_probs(model, item)
        lat.append(ms)
        ok.append(list(BANKING_AGENTS)[int(p.argmax())] == ex["target_agent"])
    bench = sum(ok[:50])
    print(f"banking | benchmark {bench}/{min(50, len(ok))} | all {sum(ok)}/{len(ok)}")
    return {"benchmark": bench, "all": sum(ok), "n": len(ok), "latency": lat_summary(lat)}


def eval_e12(model):
    model.train(False)
    rows, lat = [], []
    for it in e12:
        item = {"messages": [{"role": "user", "content": it["state"]}], "current": None, "question": it["question"],
                "options": {o: "" for o in it["options"]}}
        p, ms = timed_probs(model, item)
        lat.append(ms)
        top = int(p.argmax())
        row = {**it, "pick": it["options"][top], "ok": it["options"][top] == it["answer"], "probs": p.tolist()}
        if it["type"] == "score":
            row["level_error"] = abs(top - it["answer_index"])
            row["expected_level"] = float(sum(i * pi for i, pi in enumerate(p.tolist())))
        rows.append(row)
    by = {t: sum(r["ok"] for r in rows if r["type"] == t) for t in ["noul", "choice", "score"]}
    sc = [r for r in rows if r["type"] == "score"]
    groups = {}
    for r in rows:
        if "pair" in r:
            groups.setdefault(r["pair"], []).append(r)
    g_ok = sum(all(r["ok"] for r in g) for g in groups.values())
    exp_ok = sum(min(len(r["options"]) - 1, max(0, round(r["expected_level"]))) == r["answer_index"] for r in sc)
    print(f"E12 choice | all {sum(r['ok'] for r in rows)}/{len(rows)} | yes/no {by['noul']} | choice {by['choice']} | score {by['score']} "
          f"(within 1 level {sum(r['level_error'] <= 1 for r in sc)}/{len(sc)}) | question groups {g_ok}/{len(groups)} | "
          f"answered yes {sum(r['pick'] == 'yes' for r in rows if r['type'] == 'noul')} (truth {sum(r['answer'] == 'yes' for r in rows if r['type'] == 'noul')})")
    print(f"  score with the rounded expected level instead of the top level: {exp_ok}/{len(sc)} exact")
    print("  per category: " + ", ".join(f"{c} {sum(r['ok'] for r in rows if r['category'] == c)}/{sum(r['category'] == c for r in rows)}"
                                         for c in sorted({r['category'] for r in rows})))
    print("  mistakes:")
    for r in [r for r in rows if not r["ok"]]:
        s_ = r["state"] if len(r["state"]) <= 44 else r["state"][:41] + "..."
        print(f"    {r['id']} {r['type']:6s} | {s_:44s} | {r['question'][:36]:36s} | expected {r['answer']:14s} | got {r['pick']} ({max(r['probs']):.2f})")
    return {"all": sum(r["ok"] for r in rows), "n": len(rows), **by, "groups": g_ok, "score_expected_rule": exp_ok,
            "score_within1": sum(r["level_error"] <= 1 for r in sc), "latency": lat_summary(lat)}


def eval_ind_questions(model):
    """Independent questions: yes / no, choice, score; per type, yes / no by truth, per question wording."""
    model.train(False)
    rows, lat = [], []
    for it in ind_q:
        item = {"messages": [{"role": "user", "content": it["state"]}], "current": None, "question": it["question"],
                "options": {o: "" for o in it["options"]}}
        p, ms = timed_probs(model, item)
        lat.append(ms)
        top = int(p.argmax())
        rows.append({**it, "pick": it["options"][top], "ok": it["options"][top] == it["answer"],
                     "level_error": abs(top - it["answer_index"]) if it["type"] == "score" else None})
    by = {t: sum(r["ok"] for r in rows if r["type"] == t) for t in ["noul", "choice", "score"]}
    yn = [r for r in rows if r["type"] == "noul"]
    yes_ok = sum(r["ok"] for r in yn if r["answer"] == "yes"); n_yes = sum(r["answer"] == "yes" for r in yn)
    sc = [r for r in rows if r["type"] == "score"]
    print(f"IND questions | all {sum(r['ok'] for r in rows)}/{len(rows)} | yes/no {by['noul']}/{len(yn)} "
          f"(truth yes: {yes_ok}/{n_yes}, truth no: {by['noul'] - yes_ok}/{len(yn) - n_yes}) | choice {by['choice']}/{sum(r['type'] == 'choice' for r in rows)} "
          f"| score {by['score']}/{len(sc)} (within 1 level {sum(r['level_error'] <= 1 for r in sc)})")
    clean = [r for r in rows if not r.get("flag")]
    claim = [r for r in rows if r.get("flag") == "claim_check"]
    sc_ok = [r for r in sc if r.get("flag") != "unordered_scale"]
    print(f"  without flagged items: {sum(r['ok'] for r in clean)}/{len(clean)} | claim-check yes/no (all truth 'yes'): "
          f"{sum(r['ok'] for r in claim)}/{len(claim)} | score on proper scales: {sum(r['ok'] for r in sc_ok)}/{len(sc_ok)} "
          f"(within 1 level {sum(r['level_error'] <= 1 for r in sc_ok)}) | unordered 'scales': "
          f"{sum(r['ok'] for r in sc if r.get('flag') == 'unordered_scale')}/{len(sc) - len(sc_ok)}")
    return {"all": sum(r["ok"] for r in rows), "n": len(rows), **by, "yes_truth": yes_ok, "no_truth": by["noul"] - yes_ok,
            "unflagged": sum(r["ok"] for r in clean), "n_unflagged": len(clean), "score_proper": sum(r["ok"] for r in sc_ok),
            "latency": lat_summary(lat)}


def eval_ind_routing(model):
    """Independent conversations, two ways: LIVE (the model's previous pick is the current agent) and GIVEN (the correct
    previous agent is the current agent, so one arbitrary label cannot cascade). Reported for all turns and for the
    turns whose label is not flagged (see build_independent_test_data.py)."""
    model.train(False)
    res, lat = {}, []
    for mode in ["live", "given"]:
        rows = []
        for conv in ind_r:
            cur, prev_true = None, None
            for u in conv["user_turns"]:
                prefix = conv["messages"][:u["message_index"] + 1]
                current = cur if mode == "live" else prev_true
                item = {"messages": prefix, "current": current, "question": ROUTE_Q, "options": conv["agents"]}
                p, ms = timed_probs(model, item)
                if mode == "live":
                    lat.append(ms)
                cur = list(conv["agents"])[int(p.argmax())]
                rows.append({"type": u["turn_type"], "flag": u["flag"], "ok": cur == u["expected_agent"]})
                prev_true = u["expected_agent"]
        clean = [r for r in rows if not r["flag"]]
        res[mode] = {"all": sum(r["ok"] for r in rows), "n": len(rows), "unflagged": sum(r["ok"] for r in clean), "n_unflagged": len(clean)}
        print(f"IND routing ({mode}) | all {res[mode]['all']}/{len(rows)} | unflagged turns {res[mode]['unflagged']}/{len(clean)} | "
              + " ".join(f"{t} {sum(r['ok'] for r in rows if r['type'] == t)}/{sum(r['type'] == t for r in rows)}" for t in TURN_TYPES
                         if any(r["type"] == t for r in rows))
              + " | flagged: " + " ".join(f"{f} {sum(r['ok'] for r in rows if r['flag'] == f)}/{sum(r['flag'] == f for r in rows)}"
                                         for f in ["vague_start", "label_conflict"]))
    res["latency"] = lat_summary(lat)
    return res


def eval_ind_first(model):
    model.train(False)
    rows, lat = [], []
    for m in ind_f:
        item = {"messages": [{"role": "user", "content": m["message"]}], "current": None, "question": ROUTE_Q, "options": m["agents"]}
        p, ms = timed_probs(model, item)
        lat.append(ms)
        names = list(m["agents"])
        rows.append({"type": m["type"], "ok": names[int(p.argmax())] == m["target_agent"],
                     "top2": m["target_agent"] in [names[i] for i in p.topk(2).indices.tolist()]})
    print(f"IND first turn | top-1 {sum(r['ok'] for r in rows)}/{len(rows)} | top-2 {sum(r['top2'] for r in rows)}/{len(rows)} | "
          + " ".join(f"{k} {sum(r['ok'] for r in rows if r['type'] == k)}/{sum(r['type'] == k for r in rows)}" for k in sorted({r['type'] for r in rows})))
    return {"top1": sum(r["ok"] for r in rows), "top2": sum(r["top2"] for r in rows), "n": len(rows), "latency": lat_summary(lat)}


def buckets(rows, key, edges):
    """'n<=e' accuracy per bucket of rows[key]."""
    out, lo = [], 0
    for e in edges:
        sel = [r for r in rows if lo < r[key] <= e]
        if sel:
            out.append(f"{lo + 1}-{e}: {sum(r['ok'] for r in sel)}/{len(sel)}")
        lo = e
    return " | ".join(out)


def eval_many_questions(model):
    """Fresh many-option questions: 8 families x 30, 10-50 options."""
    model.train(False)
    rows, lat = [], []
    for it in many_q:
        item = {"messages": [{"role": "user", "content": it["state"]}], "current": None, "question": it["question"],
                "options": {o: "" for o in it["options"]}}
        p, ms = timed_probs(model, item)
        lat.append(ms)
        rows.append({"family": it["family"], "n": it["n_options"], "ok": it["options"][int(p.argmax())] == it["answer"]})
    print(f"MANY questions | {sum(r['ok'] for r in rows)}/{len(rows)} | by options: {buckets(rows, 'n', [10, 15, 20, 30, 40, 50])}")
    print("  by family: " + " | ".join(f"{f} {sum(r['ok'] for r in rows if r['family'] == f)}/{sum(r['family'] == f for r in rows)}"
                                       for f in dict.fromkeys(r["family"] for r in rows)))
    return {"all": sum(r["ok"] for r in rows), "n": len(rows), "latency": lat_summary(lat)}


def eval_many_routing(model):
    """Fresh many-agent conversations (9-15 agents): LIVE (own previous pick is the current agent) and GIVEN (the correct one)."""
    model.train(False)
    res, lat = {}, []
    for mode in ["live", "given"]:
        rows = []
        for conv in many_r:
            cur, prev_true = None, None
            for u in conv["user_turns"]:
                item = {"messages": conv["messages"][:u["message_index"] + 1], "current": cur if mode == "live" else prev_true,
                        "question": ROUTE_Q, "options": conv["agents"]}
                p, ms = timed_probs(model, item)
                if mode == "live":
                    lat.append(ms)
                cur = list(conv["agents"])[int(p.argmax())]
                rows.append({"type": u["turn_type"], "n": conv["n_agents"], "ok": cur == u["expected_agent"]})
                prev_true = u["expected_agent"]
        res[mode] = {"all": sum(r["ok"] for r in rows), "n": len(rows)}
        print(f"MANY routing ({mode}) | {res[mode]['all']}/{len(rows)} | by agents: {buckets(rows, 'n', [10, 12, 15])} | "
              + " ".join(f"{t} {sum(r['ok'] for r in rows if r['type'] == t)}/{sum(r['type'] == t for r in rows)}"
                         for t in TURN_TYPES if any(r["type"] == t for r in rows)))
    res["latency"] = lat_summary(lat)
    return res


def eval_many_first(model, reverse=False):
    """Fresh many-agent first messages (9-15 agents); reverse=True lists the agents in reverse order."""
    model.train(False)
    rows, lat, picks = [], [], []
    for m in many_f:
        names = list(m["agents"])[::-1] if reverse else list(m["agents"])
        item = {"messages": [{"role": "user", "content": m["message"]}], "current": None, "question": ROUTE_Q,
                "options": {n: m["agents"][n] for n in names}}
        p, ms = timed_probs(model, item)
        lat.append(ms)
        picks.append(names[int(p.argmax())])
        rows.append({"type": m["type"], "n": m["n_agents"], "ok": picks[-1] == m["target_agent"],
                     "top2": m["target_agent"] in [names[i] for i in p.topk(2).indices.tolist()]})
    if not reverse:
        print(f"MANY first turn | top-1 {sum(r['ok'] for r in rows)}/{len(rows)} | top-2 {sum(r['top2'] for r in rows)}/{len(rows)} | "
              f"by agents: {buckets(rows, 'n', [10, 12, 15])} | "
              + " ".join(f"{k} {sum(r['ok'] for r in rows if r['type'] == k)}/{sum(r['type'] == k for r in rows)}" for k in sorted({r['type'] for r in rows})))
    return {"top1": sum(r["ok"] for r in rows), "top2": sum(r["top2"] for r in rows), "n": len(rows), "picks": picks,
            "latency": lat_summary(lat)}


def eval_option_order(model):
    """The many-agent first messages with the agents in reverse order: the answer shouldn't depend on the position."""
    fwd, rev = eval_many_first(model), eval_many_first(model, reverse=True)
    same = sum(a == b for a, b in zip(fwd["picks"], rev["picks"]))
    print(f"OPTION ORDER | same pick with the agents reversed: {same}/{len(fwd['picks'])} | top-1 forward {fwd['top1']} / reversed {rev['top1']}")
    return {"same": same, "n": len(fwd["picks"]), "forward": fwd["top1"], "reversed": rev["top1"], "latency": rev["latency"]}


NEW_TESTS = [("MANY questions", "eval_many_questions"), ("MANY routing", "eval_many_routing"), ("MANY first turn", "eval_many_first")]


def evaluate_new(model):
    """Only the fresh tests (used for the published Signpost)."""
    res = {}
    for name, fn in [(n, globals()[f]) for n, f in NEW_TESTS]:
        log(f"[{model.key}] evaluating {name}")
        res[name] = fn(model)
        L = res[name]["latency"]
        print(f"  latency {name}: p50 {L['p50']:.0f} ms | p95 {L['p95']:.0f} ms ({L['n']} questions)\n")
    return res


ROUTING_TESTS = [("E10", "eval_e10"), ("E11", "eval_e11"), ("banking", "eval_banking"), ("IND routing", "eval_ind_routing"),
                 ("IND first turn", "eval_ind_first"), ("MANY routing", "eval_many_routing"), ("MANY first turn", "eval_many_first"),
                 ("option order", "eval_option_order")]
QUESTION_TESTS = [("E12", "eval_e12"), ("IND questions", "eval_ind_questions"), ("MANY questions", "eval_many_questions")]


def evaluate_all(model, tasks="both"):
    """Each model is scored only on the tests of the job it was trained for."""
    model.train(False)
    model.probs({"messages": [{"role": "user", "content": "warm-up"}], "current": None, "question": ROUTE_Q, "options": BANKING_AGENTS})
    res = {}
    tests = (ROUTING_TESTS if tasks in ("both", "routing") else []) + (QUESTION_TESTS if tasks in ("both", "questions") else [])
    for name, fn in [(n, globals()[f]) for n, f in tests]:
        log(f"[{model.key}] evaluating {name}")
        res[name] = fn(model)
        L = res[name]["latency"]
        print(f"  latency {name}: p50 {L['p50']:.0f} ms | p95 {L['p95']:.0f} ms | mean {L['mean']:.0f} ms ({L['n']} questions)\n")
    return res


def run_encoder(key, score_loss="plain", max_attempts=3, tasks="both"):
    """Build, train and evaluate one encoder with one score loss; free the GPU afterwards. -> results dict (also saved to JSON).
    If training collapses (see COLLAPSE_DEV), it restarts from scratch with the next seed; every attempt is recorded."""
    log(f"===== {key} / {score_loss} score loss / trained on: {tasks} =====")
    attempts = []
    for attempt in range(max_attempts):
        seed = SEED + 1000 * attempt
        random.seed(seed); torch.manual_seed(seed)              # attempt 0 = the usual seed, as in runs 19-21
        if AMP:
            torch.cuda.reset_peak_memory_stats(DEVICE)
        model = ListwiseModel(key)
        log(f"[{key}] attempt {attempt + 1} (seed {seed}) | {model.spec['name']}: {model.n_lora} LoRA layers | trainable "
            f"{sum(p.numel() for p in model.lora_params()) / 1e6:.1f} M + head {sum(p.numel() for p in model.head.parameters()) / 1e6:.2f} M | {gpu()}")
        try:
            trained = train_model(model, score_loss, seed, tasks)
            attempts.append({"seed": seed, "status": "ok"})
            break
        except Collapsed as e:
            log(f"[{key}] attempt {attempt + 1} COLLAPSED: {e} -> restarting with a new seed")
            attempts.append({"seed": seed, "status": f"collapsed: {e}"})
            del model; gc.collect()
            if AMP:
                torch.cuda.empty_cache()
    else:
        raise RuntimeError(f"{key}: training collapsed in all {max_attempts} attempts: {attempts}")
    res = {"encoder": key, "score_loss": score_loss, "tasks": tasks, "attempts": attempts, **trained}
    res.update(evaluate_all(model, tasks))
    res["peak_gpu_gb"] = torch.cuda.max_memory_allocated(DEVICE) / 1e9 if AMP else 0
    with open(f"listwise_{key}_{score_loss}_{tasks}.json", "w") as fh:
        json.dump(res, fh, indent=1)
    del model
    gc.collect()
    if AMP:
        torch.cuda.empty_cache()
    return res


RESULTS = {}

## 5 — Train and evaluate (run 26)

`TRAIN_MODE = "combined"`, ModernBERT-large, ordinal score loss, 2 epochs, seed 42, collapse guard. Then every test: the old suites and the fresh ones.

In [ ]:
if RUN_ENCODER in ENCODERS:
    for loss_name in SCORE_LOSSES:
        if TRAIN_MODE == "separate":                            # two independent specialists, same recipe and seed (run 24)
            RESULTS[f"{RUN_ENCODER} / routing only"] = run_encoder(RUN_ENCODER, loss_name, tasks="routing")
            RESULTS[f"{RUN_ENCODER} / questions only"] = run_encoder(RUN_ENCODER, loss_name, tasks="questions")
        else:
            RESULTS[f"{RUN_ENCODER} / {loss_name}"] = run_encoder(RUN_ENCODER, loss_name)

## 6 — The published Signpost on the fresh tests

`skamalj/signpost` from the Hugging Face Hub: the same encoder and head, but one pass **per option**. It's run through the same evaluation functions as above, on the same items.

In [ ]:
class SignpostHub:
    """The published Signpost (one encoder pass per option), behind the same interface as our model."""
    key = "signpost (hub)"

    def __init__(self):
        import importlib.util
        from huggingface_hub import snapshot_download
        path = snapshot_download("skamalj/signpost")
        spec = importlib.util.spec_from_file_location("signpost_hub", Path(path) / "signpost.py")
        module = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(module)
        self.sp = module.Signpost(path, device=DEVICE)

    def train(self, mode=True):
        pass

    def probs(self, item):
        p = self.sp.probs(item["messages"], item["options"], item["question"], item["current"])
        return torch.tensor(list(p.values()))


HUB_RES = {}
if RUN_SIGNPOST_HUB:
    hub = SignpostHub()
    hub.probs({"messages": [{"role": "user", "content": "warm-up"}], "current": None, "question": ROUTE_Q, "options": BANKING_AGENTS})
    HUB_RES = evaluate_new(hub)
    del hub
    gc.collect()
    if AMP:
        torch.cuda.empty_cache()

## 7 — The Strands decider on the fresh tests

[`StrandsAgents/strands-decider-2B-hobson-v19`](https://huggingface.co/StrandsAgents/strands-decider-2B-hobson-v19), zero-shot, as in `04_decider_benchmark.ipynb`. It sees all options in one prompt and has no current-agent input, so conversations are routed with the full history and with the last 3 user turns. On a T4 it runs in float32.

In [ ]:
DEC_RES = {}
if RUN_DECIDER:
    from strands_decider.infer import load_engine
    from strands_decider.schema import ChoiceQuestion

    engine = load_engine("StrandsAgents/strands-decider-2B-hobson-v19", device=DECIDER_DEVICE)
    if not torch.cuda.is_bf16_supported():
        engine.model.to(torch.float32)
    DGPU = torch.device(DECIDER_DEVICE)

    def state_text(messages, window=None):
        if window is not None:
            user_idx = [i for i, m in enumerate(messages) if m["role"] == "user"]
            messages = messages[user_idx[-window]:] if len(user_idx) > window else messages
        return "Conversation:\n" + "\n".join(f"{m['role']}: {m['content']}" for m in messages) + f"\n\nLatest user message: {messages[-1]['content']}"

    def timed(fn):
        torch.cuda.synchronize(DGPU); t0 = time.perf_counter(); out = fn(); torch.cuda.synchronize(DGPU)
        return out, (time.perf_counter() - t0) * 1000

    def decide(messages, agents, window=None):
        q = ChoiceQuestion(instructions=ROUTE_Q, criteria=dict(agents))
        return engine.ask(state_text(messages, window), {"route": q}).answers["route"].choice

    def ask_choice(it):
        q = ChoiceQuestion(instructions=it["question"], criteria={o: o for o in it["options"]})
        return engine.ask(it["state"], {"q": q}).answers["q"].choice

    decide([{"role": "user", "content": "warm-up"}], BANKING_AGENTS)

    rows, ms = [], []
    for it in many_q:
        pick, t = timed(lambda: ask_choice(it))
        ms.append(t); rows.append({"family": it["family"], "n": it["n_options"], "ok": pick == it["answer"]})
    DEC_RES["MANY questions"] = {"all": sum(r["ok"] for r in rows), "n": len(rows), "latency": lat_summary(ms)}
    print(f"decider MANY questions | {sum(r['ok'] for r in rows)}/{len(rows)} | by options: {buckets(rows, 'n', [10, 15, 20, 30, 40, 50])} | "
          f"p50 {lat_summary(ms)['p50']:.0f} ms")

    for window in [None, 3]:
        rows, ms = [], []
        for conv in many_r:
            for u in conv["user_turns"]:
                prefix = conv["messages"][:u["message_index"] + 1]
                pick, t = timed(lambda: decide(prefix, conv["agents"], window))
                ms.append(t); rows.append({"type": u["turn_type"], "n": conv["n_agents"], "ok": pick == u["expected_agent"]})
        key = "MANY routing" if window is None else "MANY routing (last 3 turns)"
        DEC_RES[key] = {"all": sum(r["ok"] for r in rows), "n": len(rows), "latency": lat_summary(ms)}
        print(f"decider {key} | {sum(r['ok'] for r in rows)}/{len(rows)} | by agents: {buckets(rows, 'n', [10, 12, 15])} | "
              + " ".join(f"{t_} {sum(r['ok'] for r in rows if r['type'] == t_)}/{sum(r['type'] == t_ for r in rows)}"
                         for t_ in TURN_TYPES if any(r["type"] == t_ for r in rows)) + f" | p50 {lat_summary(ms)['p50']:.0f} ms")

    rows, ms = [], []
    for m in many_f:
        pick, t = timed(lambda: decide([{"role": "user", "content": m["message"]}], m["agents"]))
        ms.append(t); rows.append({"n": m["n_agents"], "ok": pick == m["target_agent"]})
    DEC_RES["MANY first turn"] = {"top1": sum(r["ok"] for r in rows), "n": len(rows), "latency": lat_summary(ms)}
    print(f"decider MANY first turn | {sum(r['ok'] for r in rows)}/{len(rows)} | by agents: {buckets(rows, 'n', [10, 12, 15])} | "
          f"p50 {lat_summary(ms)['p50']:.0f} ms")

## 8 — Side by side

Every number is correct / total on the same items. **Old tests:** this run against Signpost run 25 and the decider (both from the ledger). **Fresh tests:** this run, the published Signpost and the decider, all run in this notebook, with latency (p50, ms).

In [ ]:
LW = RESULTS[f"{RUN_ENCODER} / {SCORE_LOSSES[0]}"]
SIGNPOST_R25 = {"E10 test (83)": 78, "E10 all (193)": 181, "E11 top-1 (100)": 97, "banking 50": 48, "banking 225": 210,
                "E12 (170)": 161, "IND questions (300)": 221, "IND routing live (172)": 133, "IND first turn (200)": 180}
DECIDER_REF = {"E10 test (83)": 62, "E10 all (193)": 151, "E11 top-1 (100)": 96, "banking 50": 48, "banking 225": 214,
               "E12 (170)": 160, "IND questions (300)": 253, "IND routing live (172)": 113, "IND first turn (200)": 178}
OLD = {"E10 test (83)": lambda r: r["E10"]["test"], "E10 all (193)": lambda r: r["E10"]["all"], "E11 top-1 (100)": lambda r: r["E11"]["top1"],
       "banking 50": lambda r: r["banking"]["benchmark"], "banking 225": lambda r: r["banking"]["all"], "E12 (170)": lambda r: r["E12"]["all"],
       "IND questions (300)": lambda r: r["IND questions"]["all"], "IND routing live (172)": lambda r: r["IND routing"]["live"]["all"],
       "IND first turn (200)": lambda r: r["IND first turn"]["top1"]}
print(f"{'old tests':24s} | {'listwise (run 26)':>18} | {'Signpost (run 25)':>18} | {'decider 2B':>11}")
for row, f in OLD.items():
    print(f"{row:24s} | {f(LW):>18} | {SIGNPOST_R25[row]:>18} | {DECIDER_REF[row]:>11}")


def cell(res, test, field):
    if test not in res:
        return "—"
    r = res[test]
    v = r[field]["all"] if field in ("live", "given") else r[field]
    n = r[field]["n"] if field in ("live", "given") else r["n"]
    return f"{v}/{n} · {r['latency']['p50']:.0f} ms"


NEW = [("many-option questions", "MANY questions", "all"), ("many-agent routing, live", "MANY routing", "live"),
       ("many-agent routing, previous given", "MANY routing", "given"), ("many-agent first messages", "MANY first turn", "top1")]
print(f"\n{'fresh tests':36s} | {'listwise (run 26)':>20} | {'Signpost (hub)':>20} | {'decider 2B':>20}")
for label, test, field in NEW:
    dec = cell(DEC_RES, test, "all" if field in ("live", "given") else field) if not (field == "given") else "— (no current agent)"
    print(f"{label:36s} | {cell(LW, test, field):>20} | {cell(HUB_RES, test, field):>20} | {dec:>20}")
if "MANY routing (last 3 turns)" in DEC_RES:
    r = DEC_RES["MANY routing (last 3 turns)"]
    print(f"{'  decider, last 3 user turns only':36s} | {'':>20} | {'':>20} | {r['all']}/{r['n']} · {r['latency']['p50']:.0f} ms")
oo = LW["option order"]
print(f"\noption order (listwise): same pick with agents reversed {oo['same']}/{oo['n']} | top-1 forward {oo['forward']}, reversed {oo['reversed']}")
print(f"training: {LW['train_minutes']:.1f} min, best epoch {LW['best_epoch']} | peak GPU {LW['peak_gpu_gb']:.1f} GB")